In [277]:
import requests
from bs4 import BeautifulSoup
import json
import re
import pandas as pd
import time

header = {
    'User-Agent': 
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Cookie': 'RB_PCID=1753753936265834754; spses.bb7d=*;_fcOM={"k":"ece688aff108a72b7f059b6419853e004a87322","i":"180.69.228.101.56925","r":1753753938575}; RB_SSID=gKEI5beWWG; _ga_CQHKV7VZV7=GS2.1.s1753753937$o1$g1$t1753754074$j52$l0$h0'
}

# 크롤링할 책 id
a1 = pd.read_csv('data/교보문고2225.csv')
ids = a1['kyob_code']
book_ids = ids.unique()

# 모든 책의 정보를 저장할 리스트
all_books_data = []

# 각 URL에 대해 크롤링 수행
for i, book_id in enumerate(book_ids):
    url = f'https://product.kyobobook.co.kr/detail/{book_id}'
    
    # 요청 사이에 딜레이
    time.sleep(1) 
    
    try:
        response = requests.get(url, headers = header)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        book_info = {}

        # 1. 기본정보 추출
        book_data = None
        json_ld_scripts = soup.find_all('script', type='application/ld+json')
        for script in json_ld_scripts:
            try:
                data = json.loads(script.get_text())
                if data.get('@type') == 'Book':
                    book_data = data
                    break
            except json.JSONDecodeError:
                continue

        if book_data:
            book_info['name'] = book_data.get('name')
            book_info['isbn'] = book_data.get('workExample')[0].get('isbn') if book_data.get('workExample') else None
            book_info['author'] = book_data.get('author', {}).get('name')
            book_info['genre'] = book_data.get('genre')
            book_info['datePublished'] = book_data.get('workExample')[0].get('datePublished') if book_data.get('workExample') else None
            book_info['description'] = book_data.get('description')
            book_info['publisher'] = book_data.get('publisher', {}).get('name')
            book_info['Price'] = book_data.get('workExample')[0].get('potentialAction', {}).get('expectsAcceptanceOf', {}).get('Price') if book_data.get('workExample') else None
        else:
            print(f"URL: {url} - 책 정보를 포함하는 JSON-LD를 찾을 수 없습니다. HTML에서 일부 정보 추출 시도.")
            book_info['name'] = soup.find('h1', class_='prod_title').get_text(strip=True) if soup.find('h1', class_='prod_title') else None
            book_info['isbn'] = soup.find('meta', property='books:isbn').get('content') if soup.find('meta', property='books:isbn') else None

        # 2. 추가 정보 추출
        # 저자 소개
        author_intro_tag = soup.find('div', class_='writer_info_box')
        if author_intro_tag:
            p_tag = author_intro_tag.find('p', class_='info_text')
            if p_tag:
                book_info['authorDescription'] = p_tag.get_text(separator="\n", strip=True)
            else:
                book_info['authorDescription'] = None
        else:
            book_info['authorDescription'] = None

        # 3. 무게, 쪽수
        # 사이즈는 뒤죽박죽이라 일단 제외
        basic_info_table = soup.find('table', class_='tbl_row')
        if basic_info_table:
            for row in basic_info_table.find_all('tr'):
                title = row.find('th')
                data = row.find('td')
                if title and data:
                    title_text = title.get_text(strip=True)
                    data_text = data.get_text(strip=True)
                    if '쪽수' in title_text:
                        book_info['page'] = data_text
                    elif '크기' in title_text:
                        weight_match = re.search(r'(\d+\s*g)', data_text)
                        book_info['weight'] = weight_match.group(1) if weight_match else None
        else:
            book_info['page'] = None
            book_info['weight'] = None

        # 4. 카테고리
        categories = []
        breadcrumb_list = soup.find('ol', class_='breadcrumb_list')
        if breadcrumb_list:
            for item in breadcrumb_list.find_all('li', class_='breadcrumb_item'):
                link = item.find('a', class_='btn_sub_depth')
                if link:
                    categories.append(link.get_text(strip=True))
        book_info['categories'] = " > ".join(categories) if categories else None

        # 5. 전체 리스트에 추가
        all_books_data.append(book_info)
        
        # 진행 상황
        if (i + 1) % 100 == 0: # 100개마다 진행 상황 출력
            print(f"{i + 1}권 크롤링 완료. 현재까지 {len(all_books_data)}개 데이터 수집.")

    except requests.exceptions.RequestException as e:
        print(f"URL: {url} - 요청 오류 발생: {e}")
    except Exception as e:
        print(f"URL: {url} - 데이터 처리 중 오류 발생: {e}")

# 4. 데이터프레임으로
df = pd.DataFrame(all_books_data)

# 결과 데이터프레임 확인
print("\n크롤링 완료.")
print(df.head())

URL: https://product.kyobobook.co.kr/detail/S000000450340 - 요청 오류 발생: 404 Client Error: Not Found for url: https://product.kyobobook.co.kr/detail/S000000450340
100권 크롤링 완료. 현재까지 99개 데이터 수집.
200권 크롤링 완료. 현재까지 199개 데이터 수집.
300권 크롤링 완료. 현재까지 299개 데이터 수집.


KeyboardInterrupt: 

In [273]:
# CSV 파일로 저장 (한글 깨짐 방지를 위해 encoding='utf-8-sig' 사용)
# df.to_csv("kyobobook_crawled_data.csv", index=False, encoding='utf-8-sig')
# print("데이터가 'kyobobook_crawled_data.csv' 파일로 저장되었습니다.")

In [275]:
all_books_data

[{'name': '그리움은 아무에게나 생기지 않습니다',
  'isbn': '9791196661984',
  'author': '박근혜 지음',
  'genre': '정치/사회',
  'datePublished': '20211230',
  'description': '그리움은 아무에게나 생기지 않습니다 | 서울 구치소에서의 생활이 어느덧 4년 9개월로 접어들고 있습니다. 돌아보면, 대통령으로서의 저의 시간은 언제나 긴장의 연속이었습니다. 오늘은 언제, 어디에서, 누구를 만나고, 어떤 주제로 이야기를 해야 하는지, 늘 시간을 쪼개서 일을 하면서 참……',
  'publisher': '가로세로연구소',
  'Price': '13500.0',
  'authorDescription': '저자 박근혜는 제18대 대한민국 대통령을 역임했으며, 재임 기간 동안 사드배치, 통진당 해산, 전교조 법외노조화 등을 이뤄냈다. 각종 루머와 음모로 지난 2017년 3월 10일 탄핵 판결을 받았고, 3월 31일 구속되어 총 4년 9개월의 수감 기간을 가졌다.',
  'page': '300쪽',
  'weight': '526 g',
  'categories': '국내도서 > 정치/사회 > 정치/외교 > 정치가'},
 {'name': '불편한 편의점(벚꽃 에디션)',
  'isbn': '9791161571188',
  'author': '김호연 지음',
  'genre': '소설',
  'datePublished': '20210420',
  'description': '불편한 편의점(벚꽃 에디션) | 원 플러스 원의 기쁨, 삼각김밥 모양의 슬픔, 만 원에 네 번의 폭소가 터지는 곳! 힘겨운 시대를 살아가는 우리들에게 다가온 조금 특별한 편의점 이야기2013년 세계문학상 우수상 수상작 『망원동 브라더스』로 데뷔한 후 일상적 현실을 위트 있게 그린 경……',
  'publisher': '나무옆의자',
  'Price': '15120.0',
  'authorDescription': '영화·만